In [1]:
from pathlib import Path           # imports path, a convenient way to work with file and folder paths
import os                          # imports tool for interacting with operating system, such as environment variables, directories and paths
import shutil                      # imports utilities for copying, moving or deleting files/folders
import platform                    # imports tools for getting system info such as OS, architecture, and Python version

In [2]:
workspace = Path("/workspace")

print("Python:", platform.python_version())
print("CPUs available:", os.cpu_count())

total, used, free = shutil.disk_usage(workspace)

print(f"Workspace total: {total / 1024**3:.2f} GB")
print(f"Workspace used : {used / 1024**3:.2f} GB")
print(f"Workspace free : {free / 1024**3:.2f} GB")

Python: 3.12.3
CPUs available: 128
Workspace total: 465.66 GB
Workspace used : 4.10 GB
Workspace free : 461.56 GB


In [3]:
## Defining corpus directory structure
data_root = workspace / "multilingual_medical_bert_data"        # creates main project data path          

raw_root = data_root / "raw"                                    # raw data will be stored here
processed_root = data_root / "processed"
metadata_root = data_root / "metadata"

wikimedia_raw = raw_root / "wikimedia"

for path in [
    data_root,
    raw_root,
    processed_root,
    metadata_root,
    wikimedia_raw
]:
    path.mkdir(parents=True, exist_ok=True)                     # for each path we are making the directory if it does not exist yet

print("Corpus root:")
print(data_root)

print("\nDirectories:")
for path in data_root.rglob("*"):                              # recursively look inside data look
    if path.is_dir():                                          # only prints items that are directories
        print(path)

Corpus root:
/workspace/multilingual_medical_bert_data

Directories:
/workspace/multilingual_medical_bert_data/metadata
/workspace/multilingual_medical_bert_data/processed
/workspace/multilingual_medical_bert_data/raw
/workspace/multilingual_medical_bert_data/raw/wikimedia
/workspace/multilingual_medical_bert_data/raw/wikimedia/abwiki
/workspace/multilingual_medical_bert_data/raw/wikimedia/acewiki
/workspace/multilingual_medical_bert_data/raw/wikimedia/adywiki
/workspace/multilingual_medical_bert_data/raw/wikimedia/afwiki
/workspace/multilingual_medical_bert_data/raw/wikimedia/alswiki
/workspace/multilingual_medical_bert_data/raw/wikimedia/altwiki
/workspace/multilingual_medical_bert_data/raw/wikimedia/amiwiki
/workspace/multilingual_medical_bert_data/raw/wikimedia/amwiki
/workspace/multilingual_medical_bert_data/raw/wikimedia/angwiki
/workspace/multilingual_medical_bert_data/raw/wikimedia/annwiki
/workspace/multilingual_medical_bert_data/raw/wikimedia/anpwiki
/workspace/multilingual_m

In [6]:
 %pip install -q requests tqdm pandas   #installs 3 python libraries (-q) quitely in the current jupyter environment
                                        # requests = download/fetch data from the web via http
                                        # tqdm = adds progress bars to loops/downloads

Note: you may need to restart the kernel to use updated packages.


In [7]:
import requests
import pandas as pd
from tqdm.auto import tqdm

In [8]:
url = "https://dumps.wikimedia.org/"                          # stores the wikimedia dumps address

response = requests.get(url, timeout=30)                      # send an HTTP "GET" request, gives up if server doesn't respond in 30 seconds

print("Status:", response.status_code)                        # status code : 200 means success
print("Downloaded:", len(response.content), "bytes")
print(response.text[:500])

Status: 403
Downloaded: 135 bytes
Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119. (cbba4b0)


In [9]:
# Wikimedia is rejecting anonymous/default python requests because we haven't identified ourselves
user_agent = (
    "MultilingualMedicalBERT/0.1 "
    "(https://github.com/sirohi-ml/Multilingual_medical_BERT; "
    "contact: sirohishivanshu@gmail.com)"
)

session = requests.Session()

session.headers.update({
    "User-Agent": user_agent,
    "Accept": "*/*"
})

print(dict(session.headers))

{'User-Agent': 'MultilingualMedicalBERT/0.1 (https://github.com/sirohi-ml/Multilingual_medical_BERT; contact: sirohishivanshu@gmail.com)', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive'}


In [10]:
test_response = session.get(                          # sends a GET request using the existing session, including any headers/user - agent configs etc
    "https://dumps.wikimedia.org/",
    timeout=30
)

print("Status:", test_response.status_code)
print(test_response.text[:300])

Status: 200
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en" lang="en" dir="ltr">
<!-- This file is maintained by puppet!! -->
<!-- modules/dumps/files/web/html/public_index.html -->


## Sample request to test our agent user connection

In [11]:
## Sample request to test our agent user connection

wiki_id = "simplewiki"                                              # selects simple english wikipedia
snapshot_date = "2026-09-01"                                        # selects wikipedia dump snapshot from september 1

sha_url = (                                                         # Constructs the full URL pointing to that snapshot's SHA256SUMS file.
    "https://dumps.wikimedia.org/other/"
    f"mediawiki_content_current/{wiki_id}/{snapshot_date}/"
    "xml/bzip2/SHA256SUMS"
)

response = session.get(                                             # Downloads the checksum file using our existing HTTP session.
    sha_url,
    timeout=30
)

print("URL:", response.url)
print("Status:", response.status_code)
print("Bytes:", len(response.content))
print()
print(response.text[:3000])

URL: https://dumps.wikimedia.org/other/mediawiki_content_current/simplewiki/2026-09-01/xml/bzip2/SHA256SUMS
Status: 200
Bytes: 107

bcce684428993058228430139ba5378ab6a52db306925b05faf9a23f27c63f57  simplewiki-2026-09-01-p1p1281131.xml.bz2



In [12]:
#### What is SHA256SUMS? It contains a SHA-256 hash for each dump file, 
#### which you can later use to verify that a large downloaded Wikipedia dump is complete and uncorrupted.

In [13]:
## construct the actual file URL

checksum_line = response.text.strip()                           # takes the downloaded SHA... text and removes surrounding whitespaces/newlines

sha256_hash, filename = checksum_line.split()                   # splits the line into its two components: SHA-256 hash and filename

file_url = (                                                    # builds the complete URL pointing directly to the .bz2 Wikipedia dump file
    "https://dumps.wikimedia.org/other/"
    f"mediawiki_content_current/{wiki_id}/{snapshot_date}/"
    f"xml/bzip2/{filename}"
)

print("Wiki ID :", wiki_id)
print("Snapshot:", snapshot_date)
print("File    :", filename)
print("SHA-256 :", sha256_hash)
print("URL     :", file_url)

Wiki ID : simplewiki
Snapshot: 2026-09-01
File    : simplewiki-2026-09-01-p1p1281131.xml.bz2
SHA-256 : bcce684428993058228430139ba5378ab6a52db306925b05faf9a23f27c63f57
URL     : https://dumps.wikimedia.org/other/mediawiki_content_current/simplewiki/2026-09-01/xml/bzip2/simplewiki-2026-09-01-p1p1281131.xml.bz2


In [14]:
## check the compressed file size
head_response = session.head(
    file_url,
    timeout=30,
    allow_redirects=True
)

print("Status:", head_response.status_code)

content_length = head_response.headers.get("Content-Length")

if content_length is not None:
    size_bytes = int(content_length)

    print(f"Bytes : {size_bytes:,}")
    print(f"MiB   : {size_bytes / 1024**2:.2f}")
    print(f"GiB   : {size_bytes / 1024**3:.4f}")
else:
    print("Content-Length was not provided.")

Status: 200
Bytes : 482,956,837
MiB   : 460.58
GiB   : 0.4498


In [15]:
## download simple wikipedia
from tqdm.auto import tqdm

download_dir = (                                  # assigning a directory where our data will be downloaded
    wikimedia_raw
    / wiki_id                                     # these variables are specified above
    / snapshot_date
)

download_dir.mkdir(                               # creating the directory and any missing parent directories
    parents=True,
    exist_ok=True
)

output_path = download_dir / filename             # Defines exactly where the downloaded .bz2 file will be saved.

print("Downloading to:")
print(output_path)
print()

with session.get(                                 # Opens the download as a stream, so the entire large file isn't loaded into RAM
    file_url,
    stream=True,
    timeout=(30, 300)
) as r:

    r.raise_for_status()                          # Stops with an error if the download request failed.

    total_size = int(
        r.headers.get("Content-Length", 0)
    )

    with open(output_path, "wb") as f:            # Opens the destination file in binary write mode.

        with tqdm(                                # Creates the progress bar showing downloaded byte
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=filename
        ) as progress:

            for chunk in r.iter_content(         # Downloads the file 1 MiB at a time
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(chunk)
                    progress.update(len(chunk))

print("\nDownload complete.")
print("Saved to:", output_path)
print(
    "Size:",
    f"{output_path.stat().st_size / 1024**2:.2f} MiB"
)

/workspace/multilingual_medical_bert_data/raw/wikimedia/simplewiki/2026-09-01/simplewiki-2026-09-01-p1p1281131.xml.bz2



simplewiki-2026-09-01-p1p1281131.xml.bz2:   0%|          | 0.00/461M [00:00<?, ?B/s]


Download complete.
Saved to: /workspace/multilingual_medical_bert_data/raw/wikimedia/simplewiki/2026-09-01/simplewiki-2026-09-01-p1p1281131.xml.bz2
Size: 460.58 MiB


In [16]:
## the above run satisfactorily tells us that the codes and machine are working, lets move onto the heavy lifiting now

In [17]:
## Get every wikipedia wiki ID
sitematrix_url = "https://meta.wikimedia.org/w/api.php"

params = {
    "action": "sitematrix",
    "format": "json",
    "smtype": "language",
    "smsiteprop": "url|dbname|code|sitename"
}

r = session.get(
    sitematrix_url,
    params=params,
    timeout=60
)

r.raise_for_status()

site_matrix = r.json()["sitematrix"]

wikipedia_sites = []

for key, language_group in site_matrix.items():

    if not str(key).isdigit():
        continue

    language_code = language_group.get("code")
    language_name = language_group.get("name")

    for site in language_group.get("site", []):

        dbname = site.get("dbname")
        url = site.get("url", "")

        # Keep Wikipedia projects only
        if (
            dbname
            and dbname.endswith("wiki")
            and ".wikipedia.org" in url
        ):
            wikipedia_sites.append({
                "language_code": language_code,
                "language_name": language_name,
                "wiki_id": dbname,
                "url": url
            })

wikipedia_df = (
    pd.DataFrame(wikipedia_sites)
    .drop_duplicates(subset="wiki_id")
    .sort_values("wiki_id")
    .reset_index(drop=True)
)

print("Wikipedia editions discovered:", len(wikipedia_df))

wikipedia_df.head(10)

Wikipedia editions discovered: 364


,language_code,language_name,wiki_id,url
0,aa,Qafár af,aawiki,https://aa.wikipedia.org
1,ab,аԥсшәа,abwiki,https://ab.wikipedia.org
2,ace,Acèh,acewiki,https://ace.wikipedia.org
3,ady,адыгабзэ,adywiki,https://ady.wikipedia.org
4,af,Afrikaans,afwiki,https://af.wikipedia.org
5,ak,Akan,akwiki,https://ak.wikipedia.org
6,gsw,Alemannisch,alswiki,https://als.wikipedia.org
7,alt,алтай тил,altwiki,https://alt.wikipedia.org
8,ami,Pangcah,amiwiki,https://ami.wikipedia.org
9,am,አማርኛ,amwiki,https://am.wikipedia.org


In [18]:
## discover all September 2026 dump shards
from concurrent.futures import ThreadPoolExecutor, as_completed

snapshot_date = "2026-09-01"                                  # rememeber snapshot date acts like a cutoff, all data till then and not particularly for that day

dump_base = (
    "https://dumps.wikimedia.org/other/"
    "mediawiki_content_current"
)


def get_wiki_manifest(row):
    """
    Fetch SHA256SUMS for one Wikipedia edition.
    Returns one record per compressed XML shard.
    """

    wiki_id = row["wiki_id"]

    sha_url = (
        f"{dump_base}/{wiki_id}/{snapshot_date}/"
        "xml/bzip2/SHA256SUMS"
    )

    try:
        r = session.get(
            sha_url,
            timeout=(15, 60)
        )

        # Snapshot not available for this wiki
        if r.status_code == 404:
            return []

        r.raise_for_status()

        records = []

        for line in r.text.strip().splitlines():

            if not line.strip():
                continue

            checksum, filename = line.split(maxsplit=1)

            filename = filename.strip().lstrip("*").lstrip("./")

            file_url = (
                f"{dump_base}/{wiki_id}/{snapshot_date}/"
                f"xml/bzip2/{filename}"
            )

            records.append({
                "language_code": row["language_code"],
                "language_name": row["language_name"],
                "wiki_id": wiki_id,
                "snapshot": snapshot_date,
                "filename": filename,
                "sha256": checksum,
                "file_url": file_url
            })

        return records

    except Exception as e:

        return [{
            "language_code": row["language_code"],
            "language_name": row["language_name"],
            "wiki_id": wiki_id,
            "snapshot": snapshot_date,
            "filename": None,
            "sha256": None,
            "file_url": None,
            "error": str(e)
        }]


all_records = []

rows = [
    row
    for _, row in wikipedia_df.iterrows()
]

with ThreadPoolExecutor(max_workers=3) as executor:

    futures = [
        executor.submit(
            get_wiki_manifest,
            row
        )
        for row in rows
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Checking Wikipedia dumps"
    ):
        all_records.extend(future.result())


manifest_df = pd.DataFrame(all_records)

print("Wikipedia editions checked :", len(wikipedia_df))
print(
    "Editions with dump files    :",
    manifest_df.loc[
        manifest_df["filename"].notna(),
        "wiki_id"
    ].nunique()
)

print(
    "Total compressed XML shards :",
    manifest_df["filename"].notna().sum()
)

manifest_df.head(20)

Checking Wikipedia dumps:   0%|          | 0/364 [00:00<?, ?it/s]

Wikipedia editions checked : 364
Editions with dump files    : 348
Total compressed XML shards : 393


,language_code,language_name,wiki_id,snapshot,filename,sha256,file_url
0,ady,адыгабзэ,adywiki,2026-09-01,adywiki-2026-09-01-p4p5969.xml.bz2,cd291fc85767ce3411151609769ca5cd2955df33031987...,https://dumps.wikimedia.org/other/mediawiki_co...
1,af,Afrikaans,afwiki,2026-09-01,afwiki-2026-09-01-p2p468468.xml.bz2,b3869a796c32f6955eb5d86391ab67a51ca4c158ce5371...,https://dumps.wikimedia.org/other/mediawiki_co...
2,ab,аԥсшәа,abwiki,2026-09-01,abwiki-2026-09-01-p1p55923.xml.bz2,c6481ff1737da633ae6db1b608031faf26187164f04a07...,https://dumps.wikimedia.org/other/mediawiki_co...
3,ace,Acèh,acewiki,2026-09-01,acewiki-2026-09-01-p1p33735.xml.bz2,b73607588b7a7017de9731f0b94335228d0f86d36ea167...,https://dumps.wikimedia.org/other/mediawiki_co...
4,gsw,Alemannisch,alswiki,2026-09-01,alswiki-2026-09-01-p1p94943.xml.bz2,1d248e831b7956bc48fd8600156845df9f09b7637dcddc...,https://dumps.wikimedia.org/other/mediawiki_co...
5,alt,алтай тил,altwiki,2026-09-01,altwiki-2026-09-01-p3p7701.xml.bz2,cda29290e85acdcb73f75b3668421a81cfb64859cc9c2a...,https://dumps.wikimedia.org/other/mediawiki_co...
6,ami,Pangcah,amiwiki,2026-09-01,amiwiki-2026-09-01-p3p3251.xml.bz2,1e161c2ccda8f139e8d4ca21961d5ee110265e4534e49f...,https://dumps.wikimedia.org/other/mediawiki_co...
7,am,አማርኛ,amwiki,2026-09-01,amwiki-2026-09-01-p1p55893.xml.bz2,9ed0fc5eba2176e1b4304b140280e1008441474da4a591...,https://dumps.wikimedia.org/other/mediawiki_co...
8,ang,Ænglisc,angwiki,2026-09-01,angwiki-2026-09-01-p1p28543.xml.bz2,f2408b70830749f2ed96f08eaa435e9136e814fa65e881...,https://dumps.wikimedia.org/other/mediawiki_co...
9,ann,Obolo,annwiki,2026-09-01,annwiki-2026-09-01-p1p1037.xml.bz2,fe3252efa7ce93bbdcb5a1528d8e5314f7af970e238a76...,https://dumps.wikimedia.org/other/mediawiki_co...


In [19]:
## get the size of every shard
# Keep only actual dump-file rows
files_df = (
    manifest_df[
        manifest_df["filename"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)


def get_file_size(index, url):

    try:
        r = session.head(
            url,
            timeout=(15, 60),
            allow_redirects=True
        )

        r.raise_for_status()

        size = r.headers.get("Content-Length")

        if size is None:
            return index, None, "No Content-Length"

        return index, int(size), None

    except Exception as e:
        return index, None, str(e)


results = []

with ThreadPoolExecutor(max_workers=3) as executor:

    futures = [
        executor.submit(
            get_file_size,
            idx,
            row["file_url"]
        )
        for idx, row in files_df.iterrows()
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Sizing Wikipedia shards"
    ):
        results.append(future.result())


# Add results back to dataframe
for idx, size_bytes, error in results:

    files_df.loc[idx, "size_bytes"] = size_bytes

    if error is not None:
        files_df.loc[idx, "size_error"] = error


files_df["size_mib"] = (
    files_df["size_bytes"] / 1024**2
)

files_df["size_gib"] = (
    files_df["size_bytes"] / 1024**3
)


# -------------------------------
# Summary
# -------------------------------

successful = files_df["size_bytes"].notna()

total_bytes = files_df.loc[
    successful,
    "size_bytes"
].sum()

total_gib = total_bytes / 1024**3

print("Total shards          :", len(files_df))
print("Successfully sized    :", successful.sum())
print("Failed size requests  :", (~successful).sum())

print()
print(f"Total compressed size : {total_gib:,.2f} GiB")


# Largest Wikipedias
wiki_sizes = (
    files_df
    .groupby(
        ["wiki_id", "language_code", "language_name"],
        as_index=False
    )["size_gib"]
    .sum()
    .sort_values(
        "size_gib",
        ascending=False
    )
)

print("\nLargest Wikipedia editions:\n")

display(
    wiki_sizes.head(20)
)

Sizing Wikipedia shards:   0%|          | 0/393 [00:00<?, ?it/s]

Total shards          : 393
Successfully sized    : 393
Failed size requests  : 0

Total compressed size : 146.72 GiB

Largest Wikipedia editions:



,wiki_id,language_code,language_name,size_gib
80,enwiki,en,English,43.255839
68,dewiki,de,Deutsch,10.775915
96,frwiki,fr,français,9.259023
260,ruwiki,ru,русский,6.635752
82,eswiki,es,español,6.448296
140,itwiki,it,italiano,5.588068
143,jawiki,ja,日本語,5.202409
346,zhwiki,zh,中文,4.419347
248,ptwiki,pt,português,3.680135
242,plwiki,pl,polski,3.257986


In [20]:
## save the manifest and prepare bulk download
manifest_path = (
    metadata_root /
    "wikipedia_2026_09_acquisition_manifest.csv"
)

files_df.to_csv(
    manifest_path,
    index=False
)

print("Manifest saved:")
print(manifest_path)

print()
print("Wikipedia editions :", files_df["wiki_id"].nunique())
print("Total shards       :", len(files_df))
print(
    "Compressed size    :",
    f"{files_df['size_gib'].sum():.2f} GiB"
)

Manifest saved:
/workspace/multilingual_medical_bert_data/metadata/wikipedia_2026_09_acquisition_manifest.csv

Wikipedia editions : 348
Total shards       : 393
Compressed size    : 146.72 GiB


In [21]:
import time
import hashlib
from tqdm.auto import tqdm


def download_shard(row, max_retries=5):

    wiki_id = row["wiki_id"]
    snapshot = row["snapshot"]
    filename = row["filename"]
    url = row["file_url"]

    expected_size = int(row["size_bytes"])
    expected_sha256 = row["sha256"]

    # ---------------------------------------
    # Directory
    # ---------------------------------------

    output_dir = (
        wikimedia_raw /
        wiki_id /
        snapshot
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    final_path = output_dir / filename
    partial_path = output_dir / f"{filename}.part"

    # ---------------------------------------
    # Already completed?
    # ---------------------------------------

    if final_path.exists():

        if final_path.stat().st_size == expected_size:

            return {
                "wiki_id": wiki_id,
                "filename": filename,
                "status": "already_downloaded",
                "path": str(final_path)
            }

    # ---------------------------------------
    # Resume partial download
    # ---------------------------------------

    existing_size = (
        partial_path.stat().st_size
        if partial_path.exists()
        else 0
    )

    for attempt in range(1, max_retries + 1):

        try:

            headers = {}

            if existing_size > 0:
                headers["Range"] = f"bytes={existing_size}-"

            with session.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 300)
            ) as r:

                # Server accepted resume
                if existing_size > 0 and r.status_code == 206:
                    mode = "ab"

                # Server ignored Range -> restart file
                elif r.status_code == 200:
                    mode = "wb"
                    existing_size = 0

                else:
                    r.raise_for_status()

                with open(partial_path, mode) as f:

                    for chunk in r.iter_content(
                        chunk_size=8 * 1024 * 1024
                    ):

                        if chunk:
                            f.write(chunk)

            # ---------------------------------------
            # Check size
            # ---------------------------------------

            actual_size = partial_path.stat().st_size

            if actual_size != expected_size:

                raise RuntimeError(
                    f"Size mismatch: "
                    f"{actual_size:,} != "
                    f"{expected_size:,}"
                )

            # ---------------------------------------
            # SHA-256 verification
            # ---------------------------------------

            hasher = hashlib.sha256()

            with open(partial_path, "rb") as f:

                while True:

                    block = f.read(
                        16 * 1024 * 1024
                    )

                    if not block:
                        break

                    hasher.update(block)

            actual_sha256 = hasher.hexdigest()

            if actual_sha256 != expected_sha256:

                raise RuntimeError(
                    "SHA256 mismatch"
                )

            # Atomic completion
            partial_path.rename(final_path)

            return {
                "wiki_id": wiki_id,
                "filename": filename,
                "status": "downloaded",
                "path": str(final_path)
            }

        except Exception as e:

            if attempt == max_retries:

                return {
                    "wiki_id": wiki_id,
                    "filename": filename,
                    "status": "failed",
                    "error": str(e)
                }

            # Refresh partial size before retry
            existing_size = (
                partial_path.stat().st_size
                if partial_path.exists()
                else 0
            )

            time.sleep(
                min(2 ** attempt, 30)
            )

In [ ]:
## download all 146 GiB
download_rows = [
    row
    for _, row in files_df.iterrows()
]

download_results = []

total_bytes = int(
    files_df["size_bytes"].sum()
)

status_path = (
    metadata_root /
    "wikipedia_download_status.csv"
)

print(f"Wikipedia editions : {files_df['wiki_id'].nunique()}")
print(f"Total shards       : {len(files_df)}")
print(f"Total download     : {total_bytes / 1024**3:.2f} GiB")
print()
print("Starting bulk download with 3 workers...")
print()


with tqdm(
    total=total_bytes,
    unit="B",
    unit_scale=True,
    unit_divisor=1024,
    desc="Wikipedia"
) as progress:

    with ThreadPoolExecutor(max_workers=3) as executor:

        future_to_row = {
            executor.submit(
                download_shard,
                row
            ): row

            for row in download_rows
        }

        for future in as_completed(future_to_row):

            row = future_to_row[future]

            try:
                result = future.result()

            except Exception as e:

                result = {
                    "wiki_id": row["wiki_id"],
                    "filename": row["filename"],
                    "status": "failed",
                    "error": str(e)
                }

            # Add useful metadata
            result["snapshot"] = row["snapshot"]
            result["size_bytes"] = int(row["size_bytes"])
            result["size_gib"] = row["size_gib"]

            download_results.append(result)

            # Advance global progress
            if result["status"] in {
                "downloaded",
                "already_downloaded"
            }:
                progress.update(
                    int(row["size_bytes"])
                )

            # Save checkpoint after every shard
            pd.DataFrame(
                download_results
            ).to_csv(
                status_path,
                index=False
            )


print("\nBulk download pass complete.")

Wikipedia editions : 348
Total shards       : 393
Total download     : 146.72 GiB

Starting bulk download with 3 workers...



Wikipedia:   0%|          | 0.00/147G [00:00<?, ?B/s]

In [24]:
results_df = pd.DataFrame(download_results)

print("\nDownload summary")
print("----------------")

print(
    results_df["status"]
    .value_counts(dropna=False)
)

successful = results_df[
    results_df["status"].isin(
        ["downloaded", "already_downloaded"]
    )
]

downloaded_gib = (
    successful["size_bytes"].sum()
    / 1024**3
)

print()
print(
    f"Verified data : {downloaded_gib:.2f} GiB"
)

print(
    f"Verified shards: "
    f"{len(successful)}/{len(files_df)}"
)

failed = results_df[
    results_df["status"] == "failed"
]

if len(failed) > 0:

    print(
        f"\nFailed shards: {len(failed)}"
    )

    display(
        failed[
            [
                "wiki_id",
                "filename",
                "error"
            ]
        ]
    )


Download summary
----------------
status
downloaded            364
already_downloaded     28
failed                  1
Name: count, dtype: int64

Verified data : 146.62 GiB
Verified shards: 392/393

Failed shards: 1


,wiki_id,filename,error
31,bawiki,bawiki-2026-09-01-p1p201432.xml.bz2,416 Client Error: Requested Range Not Satisfia...


In [25]:
from pathlib import Path

bawiki_dir = Path(
    "/workspace/multilingual_medical_bert_data/raw/wikimedia/bawiki/2026-09-01"
)

filename = "bawiki-2026-09-01-p1p201432.xml.bz2"

final_file = bawiki_dir / filename
part_file = bawiki_dir / f"{filename}.part"

print("Final exists :", final_file.exists())
print("Part exists  :", part_file.exists())

if part_file.exists():
    print(
        "Partial size :",
        round(part_file.stat().st_size / 1024**2, 2),
        "MiB"
    )

Final exists : False
Part exists  : True
Partial size : 105.79 MiB


In [26]:
part_file.unlink()

print("Deleted stale partial:", not part_file.exists())

Deleted stale partial: True


In [28]:
filename = "bawiki-2026-09-01-p1p201432.xml.bz2"

bawiki_row = manifest_df[
    manifest_df["filename"] == filename
].iloc[0].copy()

# Get expected size directly from the server
head = session.head(
    bawiki_row["file_url"],
    timeout=60,
    allow_redirects=True
)

head.raise_for_status()

bawiki_row["size_bytes"] = int(
    head.headers["Content-Length"]
)

print("File :", filename)
print(
    "Expected size:",
    round(bawiki_row["size_bytes"] / 1024**2, 2),
    "MiB"
)
print("SHA256:", bawiki_row["sha256"])

File : bawiki-2026-09-01-p1p201432.xml.bz2
Expected size: 105.79 MiB
SHA256: a3ce08877a2eced6c003b1d881bd6ac771dbec4dac9f66a88df8906d929e87a7


In [29]:
result = download_shard(
    bawiki_row,
    max_retries=5
)

result

{'wiki_id': 'bawiki',
 'filename': 'bawiki-2026-09-01-p1p201432.xml.bz2',
 'status': 'downloaded',
 'path': '/workspace/multilingual_medical_bert_data/raw/wikimedia/bawiki/2026-09-01/bawiki-2026-09-01-p1p201432.xml.bz2'}

In [30]:
from pathlib import Path

raw = Path(
    "/workspace/multilingual_medical_bert_data/raw/wikimedia"
)

downloaded_files = sorted(
    raw.rglob("*.xml.bz2")
)

partial_files = sorted(
    raw.rglob("*.xml.bz2.part")
)

total_bytes = sum(
    p.stat().st_size
    for p in downloaded_files
)

print("Completed shards :", len(downloaded_files))
print("Partial downloads:", len(partial_files))
print(
    "Total raw size   :",
    round(total_bytes / 1024**3, 2),
    "GiB"
)

if len(downloaded_files) == 393 and len(partial_files) == 0:
    print("\n✓ Wikipedia acquisition COMPLETE")
else:
    print("\n⚠ Check acquisition state")

Completed shards : 393
Partial downloads: 0
Total raw size   : 146.72 GiB

✓ Wikipedia acquisition COMPLETE


In [8]:
# %pip install -q beautifulsoup4

In [10]:
## Inspect the mediawiki_content_current root
from bs4 import BeautifulSoup                        # Be...soup is used to parse and navigate HTML

current_url = (                                      # stores the URL of wkm's 'mediawiki_content_current' dump directory
    "https://dumps.wikimedia.org/other/"
    "mediawiki_content_current/readme.html"
)

response = session.get(                              # Downloads the page using our existing session (which likely already contains our User-Agent).
    current_url,
    timeout=30
)

response.raise_for_status()                          # raises an error if the request failed

print("Status:", response.status_code)
print("Downloaded:", len(response.content), "bytes")

soup = BeautifulSoup(response.text, "html.parser")   # Converts the raw HTML into a structure Python can easily search.

# links = [
#     a.get("href")                                    # Ignores <a> elements that don't have an href.
#     for a in soup.find_all("a")                      # Finds every HTML anchor (<a>) element, i.e. every link.
#     if a.get("href")
# ]

links = [
    {
        "text": a.get_text(strip=True),
        "href": a.get("href")
    }
    for a in soup.find_all("a")
    if a.get("href")
]

links

links[:50]                                           # Displays the first 50 extracted links.

HTTPError: 403 Client Error: Forbidden for url: https://dumps.wikimedia.org/other/mediawiki_content_current/readme.html

In [11]:
wiki_id = "simplewiki"
snapshot_date = "2026-09-01"

sha_url = (
    f"https://dumps.wikimedia.org/other/"
    f"mediawiki_content_current/"
    f"{wiki_id}/"
    f"{snapshot_date}/"
    f"xml/bzip2/SHA256SUMS"
)

print(sha_url)

response = session.get(
    sha_url,
    timeout=30
)

print("Status:", response.status_code)
print("Bytes:", len(response.content))

if response.status_code == 200:
    print("\nExport is complete.\n")
    print(response.text[:3000])
elif response.status_code == 404:
    print("\nThis snapshot is not available/complete yet.")
else:
    print(response.text[:500])

https://dumps.wikimedia.org/other/mediawiki_content_current/simplewiki/2026-09-01/xml/bzip2/SHA256SUMS
Status: 403
Bytes: 135
Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119. (cbba4b0)


In [12]:
print("Session headers:")
print(dict(session.headers))

Session headers:
{'User-Agent': 'python-requests/2.32.5', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive'}
